In [ ]:
# run this notebook after 24_missense_lof_python
# use this notebook to generate genotype tables for the burden test
# after this, run 26_methods_ancestry_R

In [ ]:
library(data.table)
library(dplyr)
library(bedtoolsr)
library(arrow)
library(jsonlite)
library(stringr)

In [ ]:
gt = fread("./vat_repair_genes_missense_lof_trio_parents_siblings_nonref_GT.tsv", sep = "\t", header=TRUE)
gt = as.data.frame(gt)

In [ ]:
# read in parents and siblings 
trios = fread("./relatedness_dec20/trios_final.txt", header=TRUE)
parents = c(trios$par1, trios$par2)
sibs =fread("./relatedness_dec20/sibs_final.txt", header=TRUE)
par_sibs = unique(c(parents, sibs$idv1, sibs$idv2))
length(par_sibs)

In [ ]:
gt = gt[,which(names(gt) %in% c("locus", "alleles", par_sibs))]

In [ ]:
genes = fread("vat_repair_genes_deleterious_missense_lof_final.tsv", sep = "\t", header=TRUE)
genes = genes %>% filter(is_canonical_transcript)
genes = unique(genes)
genes = genes %>% filter(gvs_all_an/max(gvs_all_an) >= 0.99)
gene_list = unique(genes$gene_symbol)

In [ ]:
lof_description = c('transcript_ablation', 'splice_acceptor_variant', 
                    'splice_donor_variant', 'stop_gained','frameshift_variant')
missense_description = c("missense_variant")

In [ ]:
desc_list  = strsplit(genes$consequence, ",", fixed = TRUE)
revel_num  = as.numeric(genes$revel)

# lof: any term in lof_description
genes$lof = vapply(
  desc_list,
  function(d) any(d %in% lof_description),
  logical(1)
)

# missense: must contain "missense_variant" AND REVEL > 0.5
has_missense = vapply(
  desc_list,
  function(d) "missense_variant" %in% d,
  logical(1)
)

genes$missense = has_missense & revel_num > 0.75

In [ ]:
lof_list = genes %>% filter(lof) %>% filter(gvs_all_af <= 0.01)
missense_list = genes %>% filter(missense)
all_list = rbind(lof_list, missense_list)
gene_list = unique(all_list$gene_symbol)

In [ ]:
fwrite(all_list, "./repair_genes_lof_missense_variants.txt", sep = "\t", row.names = FALSE, quote = FALSE)

In [ ]:
parse_vec = function(s) {
  s = trimws(s)
  if (is.na(s) || s == "") return(character())
  fromJSON(s)  # expects e.g. ["A","G"]
}

In [ ]:
sibs = data.frame(idv1 = sibs$idv1, idv2 = sibs$idv2, fid = sibs$fid)
sibs[gene_list] = 0
trios = as.data.frame(trios)
trios[gene_list] = 0
lof_list$alleles = paste0('["', lof_list$ref_allele, '","', lof_list$alt_allele, '"]')
lof_list$locus = paste(lof_list$contig, lof_list$position, sep =":")
missense_list$alleles = paste0('["', missense_list$ref_allele, '","', missense_list$alt_allele, '"]')
missense_list$locus = paste(missense_list$contig, missense_list$position, sep =":")

In [ ]:
lof_gts = merge(lof_list, gt, by = c("locus", "alleles"), all.x =TRUE)
missense_gts = merge(missense_list, gt, by = c("locus", "alleles"), all.x=TRUE)

In [ ]:
gt_number = function(gt_raw){
    if (is.na(gt_raw)){
        return(0)
    } else if (gt_raw %in% c("0/0", "0|0")){
        return(0)
    } else if (gt_raw %in% c("1/1","1|1")){
        return(2)
    } else if (gt_raw %in% c("0/1", "0|1", "1/0", "1|0")){
        return (1)
    } else {
        return(NA)
    }
}

In [ ]:
sum_gt_trio = function(gts){
    s = 0
    for (m in 1:length(gts)){
        s = s + gt_number(gts[m])
    }
    return(s)
}

In [ ]:
# go through trios: lof
trios_lof = trios
lof_gts = lof_gts %>% filter(nchar(ref_allele) == 1 & nchar(alt_allele) == 1)
col_parents = which(names(lof_gts) %in% c("locus", "alleles", parents, "gene_symbol"))
lof_gts_parents = lof_gts[,..col_parents]
for (i in 1:nrow(lof_gts_parents)){
    col_ids = which(!is.na(lof_gts_parents[i,]) & 
                    lof_gts_parents[i,] %in% c("0/1", "0|1", "1/0", "1|0", "1/1", "1|1"))
    if (length(col_ids) != 0){
        for (j in 1:length(col_ids)){
            # which idv 
            idv = names(lof_gts_parents)[col_ids[j]]
            id_idx = col_ids[j]
            geno_num = gt_number(lof_gts_parents[i,..id_idx][[1]])
            # all trios where this idv is a parent 
            trio_idv = which(trios_lof$par1 == idv | trios_lof$par2 == idv)
            # column for this gene
            gene_name = lof_gts_parents$gene_symbol[i]
            gene_col = which(names(trios_lof) == gene_name)
            # add number to trio's column for this gene 
            trios_lof[trio_idv,gene_col]= trios_lof[trio_idv,gene_col] + geno_num 
        }
    }
}

In [ ]:
# go through trios: missense
trios_missense = trios
col_parents = which(names(missense_gts) %in% c("locus", "alleles", parents, "gene_symbol"))
missense_gts_parents = missense_gts[,..col_parents]
for (i in 1:nrow(missense_gts_parents)){
    col_ids = which(!is.na(missense_gts_parents[i,]) & 
                    missense_gts_parents[i,] %in% c("0/1", "0|1", "1/0", "1|0", "1/1", "1|1"))
    if (length(col_ids) != 0){
        for (j in 1:length(col_ids)){
            # which idv 
            idv = names(missense_gts_parents)[col_ids[j]]
            id_idx = col_ids[j]
            geno_num = gt_number(missense_gts_parents[i,..id_idx][[1]])
            # all trios where this idv is a parent 
            trio_idv = which(trios_missense$par1 == idv | trios_missense$par2 == idv)
            # column for this gene
            gene_name = missense_gts_parents$gene_symbol[i]
            gene_col = which(names(trios_missense) == gene_name)
            # add number to trio's column for this gene 
            trios_missense[trio_idv,gene_col]= trios_missense[trio_idv,gene_col] + geno_num 
        }
    }
}
trios_missense$id = paste(trios_missense$par1, trios_missense$par2, trios_missense$off, sep = "_")

In [ ]:
trios_lof$id = paste(trios_lof$par1, trios_lof$par2, trios_lof$off, sep = "_")
trios_missense$id = paste(trios_missense$par1, trios_missense$par2, trios_missense$off, sep = "_")

In [ ]:
# read 96-types 
cosmic_vectors = fread("./families_COSMIC_96_mutation_rates.txt", sep = "\t", header=TRUE)

In [ ]:
trios_lof_with_vector = merge(trios_lof, cosmic_vectors, by = c("fid"))
trios_missense_with_vector = merge(trios_missense, cosmic_vectors, by = c("fid"))
trios_lof_with_vector$vector_type = "lof"
trios_missense_with_vector$vector_type = "missense"

In [ ]:
# write trio_lofs and trio_missense 
fwrite(trios_lof_with_vector, "./aou_trios_repair_gene_lof_96_COSMIC.txt", sep = "\t", quote=FALSE, row.names=FALSE)
fwrite(trios_missense_with_vector, "./aou_trios_repair_gene_missense_96_COSMIC.txt", sep = "\t", quote=FALSE, row.names=FALSE)

In [ ]:
snipar_out = fread("./ibd_all_dec20.txt", sep = "\t", header=TRUE)
snipar_out$chrom = paste("chr", snipar_out$Chr, sep = "")

In [ ]:
calculate_genotype_table = function(gts){
    setDT(gts)
    setDT(sibs)
    setDT(snipar_out)

    # ── 1. Precompute shared objects ─────────────────────────────────────────────

    gts[, af := gvs_all_af]

    # var_dt for foverlaps — build ONCE
    var_dt = data.table(
      chrom = gts$contig,
      start = gts$position,
      end   = gts$position + 1L
    )
    var_dt[, idx := .I]
    setkey(var_dt, chrom, start, end)

    # ── 2. Vectorized gt_number ─────────────────────────────────────────────────

    gt_number_vec = function(x) {
      out = rep(NA_integer_, length(x))
      out[is.na(x)]                              = 0L
      out[x %in% c("0/0", "0|0")]               = 0L
      out[x %in% c("0/1", "0|1", "1/0", "1|0")] = 1L
      out[x %in% c("1/1", "1|1")]               = 2L
      out
    }

    # ── 3. Vectorized impute_no_ibd ─────────────────────────────────────────────
    # Fallback when no IBD info is available or IBD-based computation fails.
    # g1, g2: integer genotype vectors (0/1/2); f: allele frequency vector

    impute_no_ibd_vec = function(g1, g2, f) {
      add = rep(NA_real_, length(g1))
      ok  = !is.na(g1) & !is.na(g2)

      # (0,0)
      m = ok & g1 == 0L & g2 == 0L
      if (any(m)) add[m] = 2 * f[m] / (2 - f[m])

      # (1,0) or (0,1)
      m = ok & ((g1 == 1L & g2 == 0L) | (g1 == 0L & g2 == 1L))
      if (any(m)) add[m] = 2 / (2 - f[m])

      # (2,0) or (0,2)
      m = ok & ((g1 == 2L & g2 == 0L) | (g1 == 0L & g2 == 2L))
      if (any(m)) add[m] = 2

      # (1,1)
      m = ok & g1 == 1L & g2 == 1L
      if (any(m)) add[m] = 2 + (2 * f[m] - 1) / (1 + f[m] * (1 - f[m]))

      # (2,1) or (1,2)
      m = ok & ((g1 == 2L & g2 == 1L) | (g1 == 1L & g2 == 2L))
      if (any(m)) add[m] = 2 * (1 + 2 * f[m]) / (1 + f[m])

      # (2,2)
      m = ok & g1 == 2L & g2 == 2L
      if (any(m)) add[m] = 2 * (1 + 3 * f[m]) / (1 + f[m])

      add
    }

    # ── 4. Build sibs_lof summary table ─────────────────────────────────────────

    sibs_lof = copy(sibs)
    sibs_lof[, id := paste(idv1, idv2, sep = "_")]
    sibs_lof = sibs_lof[, .(ids = list(id)), by = fid]

    gene_cols = unique(gts$gene_symbol)
    gene_cols = gene_cols[!is.na(gene_cols)]
    sibs_lof[, (gene_cols) := 0]

    # ── 5. Precompute constant vectors ──────────────────────────────────────────

    n_variants = nrow(gts)
    af_vec   = gts$af
    p_vec    = 1 - af_vec
    gene_vec = gts$gene_symbol

    # ── 6. Main loop ────────────────────────────────────────────────────────────

    for (i in seq_len(nrow(sibs_lof))) {
    #for (i in 1:1){
      if (i %% 100 == 0L) message(i)

      sibpairs = sibs_lof$ids[[i]]

      # Per-variant trackers across all sib pairs in this family
      has_ibd0        = logical(n_variants)
      add_ibd0        = rep(NA_real_, n_variants)
      max_add_ibd1    = rep(NA_real_, n_variants)
      max_add_ibd2    = rep(NA_real_, n_variants)
      max_add_imputed = rep(NA_real_, n_variants)   # lowest-priority fallback

      for (s in seq_along(sibpairs)) {
        idvs = strsplit(sibpairs[s], "_")[[1]]
        id1 = idvs[1]
        id2 = idvs[2]

        # IBD segments for this pair
        pair_ibd = snipar_out[ID1 == id1 & ID2 == id2]

        # Genotypes for both sibs (needed for both IBD and imputation paths)
        gt1_all = gt_number_vec(gts[[id1]])
        gt2_all = gt_number_vec(gts[[id2]])

        # ── Case A: No IBD data for this pair → impute all variants ──────────
        if (nrow(pair_ibd) == 0L) {
          imp = impute_no_ibd_vec(gt1_all, gt2_all, af_vec)
          has_imp = !is.na(imp)
          if (any(has_imp)) {
            max_add_imputed[has_imp] = pmax(max_add_imputed[has_imp], imp[has_imp], na.rm = TRUE)
          }
          next
        }

        # ── Case B: Has IBD data → annotate and compute ──────────────────────
        ibd_annot = rep(NA_integer_, n_variants)

        for (t in 0:2) {
          ibd_t = pair_ibd[IBDType == t, .(chrom, start = start_coordinate, end = stop_coordinate)]
          if (nrow(ibd_t) == 0L) next
          setkey(ibd_t, chrom, start, end)
          hits = foverlaps(var_dt, ibd_t, type = "any", which = TRUE, nomatch = 0L)
          if (nrow(hits)) ibd_annot[var_dt$idx[hits$xid]] = t
        }

        valid = which(!is.na(ibd_annot))

        # ── IBD-based computation for annotated variants ─────────────────────
        add_full = rep(NA_real_, n_variants)

        if (length(valid) > 0L) {
          gt1  = gt1_all[valid]
          gt2  = gt2_all[valid]
          q    = af_vec[valid]
          p    = p_vec[valid]
          ibdv = ibd_annot[valid]

          add = rep(NA_real_, length(valid))

          # ── IBD = 0 ────────────────────────────────────────────────────────
          idx0 = ibdv == 0L & !is.na(gt1) & !is.na(gt2)
          if (any(idx0)) add[idx0] = gt1[idx0] + gt2[idx0]

          # ── IBD = 1 ────────────────────────────────────────────────────────
          idx1 = ibdv == 1L & !is.na(gt1) & !is.na(gt2)
          if (any(idx1)) {
            a1 = gt1[idx1]; a2 = gt2[idx1]
            q1 = q[idx1];   p1 = p[idx1]

            good = !is.na(a1) & !is.na(a2) & abs(a1 - a2) != 2L
            add1 = rep(NA_real_, sum(idx1))

            if (any(good)) {
              ag1 = a1[good]; ag2 = a2[good]
              qg  = q1[good];  pg = p1[good]
              a = rep(NA_real_, sum(good))

              m = ag1 == 0L & ag2 == 0L
              if (any(m)) a[m] = qg[m]

              m = (ag1 == 0L & ag2 == 1L) | (ag1 == 1L & ag2 == 0L)
              if (any(m)) a[m] = 1 + qg[m]

              m = ag1 == 1L & ag2 == 1L
              if (any(m)) {
                a[m] = 1 + 2*qg[m]
              }

              m = (ag1 == 1L & ag2 == 2L) | (ag1 == 2L & ag2 == 1L)
              if (any(m)) a[m] = 2 + qg[m]

              m = ag1 == 2L & ag2 == 2L
              if (any(m)) a[m] = 3 + qg[m]

              add1[good] = a
            }
            # add1 remains NA for !good variants → will be imputed below
            add[idx1] = add1
          }

          # ── IBD = 2 ────────────────────────────────────────────────────────
          idx2 = ibdv == 2L & !is.na(gt1) & !is.na(gt2)
          if (any(idx2)) {
            same = !is.na(gt1[idx2]) & !is.na(gt2[idx2]) & gt1[idx2] == gt2[idx2]
            add2 = rep(NA_real_, sum(idx2))
            if (any(same)) add2[same] = gt1[idx2][same] + 2 * q[idx2][same]
            add[idx2] = add2
          }

          add_full[valid] = add
        }

        # ── Accumulate IBD-based values into trackers ────────────────────────
        ibd_full = rep(NA_integer_, n_variants)
        if (length(valid) > 0L) ibd_full[valid] = ibd_annot[valid]

        # IBD-0: track running max (like IBD-1 and IBD-2)
        is0 = !is.na(ibd_full) & ibd_full == 0L & !is.na(add_full)
        if (any(is0)) {
          has_ibd0[is0] = TRUE
          add_ibd0[is0] = pmax(add_ibd0[is0], add_full[is0], na.rm = TRUE)
        }

        # IBD-1: track running max
        is1 = !is.na(ibd_full) & ibd_full == 1L & !is.na(add_full)
        if (any(is1)) {
          max_add_ibd1[is1] = pmax(max_add_ibd1[is1], add_full[is1], na.rm = TRUE)
        }

        # IBD-2: track running max
        is2 = !is.na(ibd_full) & ibd_full == 2L & !is.na(add_full)
        if (any(is2)) {
          max_add_ibd2[is2] = pmax(max_add_ibd2[is2], add_full[is2], na.rm = TRUE)
        }

        # ── Impute variants where IBD-based computation failed or was absent ─
        # Covers: is.na(ibd_annot), IBD=1 failed good, IBD=2 mismatch, NA gts
        need_impute = is.na(add_full)
        if (any(need_impute)) {
          imp = impute_no_ibd_vec(gt1_all[need_impute], gt2_all[need_impute], af_vec[need_impute])
          has_imp = !is.na(imp)
          if (any(has_imp)) {
            idx_imp = which(need_impute)[has_imp]
            max_add_imputed[idx_imp] = pmax(max_add_imputed[idx_imp], imp[has_imp], na.rm = TRUE)
          }
        }
      }

      # ── Compute add_family with priority: IBD0 > IBD1/IBD2 > imputed ──────
      add_family = rep(NA_real_, n_variants)

      # Priority 1: any IBD-0 → use that value
      add_family[has_ibd0] = add_ibd0[has_ibd0]

      # Priority 2: no IBD-0 → compare max IBD-1 vs max IBD-2
      no0 = !has_ibd0
      both  = no0 & !is.na(max_add_ibd1) & !is.na(max_add_ibd2)
      only1 = no0 & !is.na(max_add_ibd1) &  is.na(max_add_ibd2)
      only2 = no0 &  is.na(max_add_ibd1) & !is.na(max_add_ibd2)

      if (any(only1)) add_family[only1] = max_add_ibd1[only1]
      if (any(only2)) add_family[only2] = max_add_ibd2[only2]

      if (any(both)) {
        int1 = as.integer(floor(abs(max_add_ibd1[both])))
        int2 = as.integer(floor(abs(max_add_ibd2[both])))
        tied = int1 == int2
        val = ifelse(tied, max_add_ibd1[both],
                      pmax(max_add_ibd1[both], max_add_ibd2[both]))
        add_family[both] = val
      }

      # Priority 3 (lowest): no IBD-based value available → use imputed
      use_imputed = is.na(add_family) & !is.na(max_add_imputed)
      if (any(use_imputed)) add_family[use_imputed] = max_add_imputed[use_imputed]

      # ── Aggregate per gene and update sibs_lof ──────────────────────────────
      agg_dt = data.table(gene_symbol = gene_vec, add_family = add_family)
      agg = agg_dt[!is.na(gene_symbol) & !is.na(add_family),
                    .(score = sum(add_family)), by = gene_symbol]

      if (nrow(agg)) {
        gs = agg$gene_symbol
        sc = agg$score
        current = as.numeric(sibs_lof[i, ..gs])
        sibs_lof[i, (gs) := as.list(current + sc)]
      }
    }
    return(sibs_lof)
}


In [ ]:
sibs_lof = calculate_genotype_table(lof_gts)

In [ ]:
sibs_missense = calculate_genotype_table(missense_gts)

In [ ]:
sibs_lof_with_vector = merge(sibs_lof, cosmic_vectors, by = c("fid"))
sibs_missense_with_vector = merge(sibs_missense, cosmic_vectors, by = c("fid"))
sibs_lof_with_vector$vector_type = "lof"
sibs_missense_with_vector$vector_type = "missense"
fwrite(sibs_lof_with_vector, "./aou_sibs_repair_gene_lof_96_COSMIC.txt", sep = "\t", quote=FALSE, row.names=FALSE)
fwrite(sibs_missense_with_vector, "./aou_sibs_repair_gene_lof_96_COSMIC.txt", sep = "\t", quote=FALSE, row.names=FALSE)

In [ ]:
# combine across families
families = fread("./families_COSMIC_96_mutation_rates.txt")

In [ ]:
genes = fread("vat_repair_genes_deleterious_missense_lof_final.tsv", sep = "\t", header=TRUE)
genes = genes %>% filter(is_canonical_transcript)
genes = unique(genes)
genes = genes %>% filter(gvs_all_an/max(gvs_all_an) >= 0.99)
gene_list = unique(genes$gene_symbol)
extract_names = c("fid", gene_list)

In [ ]:
sib_counts = fread("./relatedness_dec20/sibs_passing_qc_with_parental_ages.txt") %>% select(mother_age, father_age, fid)
trio_counts = fread("./relatedness_dec20/trio_counts.txt") %>% select(mother_age, father_age, fid)
safe_mean = function(x) if (all(is.na(x))) NA_real_ else mean(x, na.rm = TRUE)

trio_counts = trio_counts %>%
  group_by(fid) %>%
  summarise(
    mother_age = safe_mean(mother_age),
    father_age = safe_mean(father_age),
    .groups = "drop"
  )

sib_counts = sib_counts %>%
  group_by(fid) %>%
  summarise(
    mother_age = safe_mean(mother_age),
    father_age = safe_mean(father_age),
    .groups = "drop"
  )

In [ ]:
# now read in trio lof (indels + snps)
trio_lof = fread("./aou_trios_repair_gene_lof_96_COSMIC.txt")
to_extract = extract_names[which(extract_names %in% names(trio_lof))]
trio_lof = trio_lof[,..to_extract]
trio_lof$source = "trio"
trio_lof = merge(trio_lof, trio_counts, by = "fid", all.x = TRUE)

# and read in sib lof (indels + snps)
sib_lof = fread("./aou_sibs_repair_gene_lof_96_COSMIC.txt")
to_extract = extract_names[which(extract_names %in% names(sib_lof))]
sib_lof = sib_lof[,..to_extract]
sib_lof$source = "sib"
sib_lof = merge(sib_lof, sib_counts, by = "fid", all.x = TRUE)


# if fid in both sib and trio, remove from sib 
sib_lof = sib_lof[which(!sib_lof$fid %in% trio_lof$fid),]
all_lof = rbind(unique(trio_lof), sib_lof)
nrow(all_lof)
families_lof = merge(families, all_lof, by = "fid", all.x = TRUE)
nrow(families_lof)

In [ ]:
# and read in sib lof (indels + snps)
sib_lof = fread("./aou_sibs_repair_gene_missense_96_COSMIC.txt")
to_extract = extract_names[which(extract_names %in% names(sib_lof))]
sib_lof = sib_lof[,..to_extract]
sib_lof$source = "sib"
sib_lof = merge(sib_lof, sib_counts, by = "fid", all.x = TRUE)


trio_lof = fread("./aou_trios_repair_gene_missense_96_COSMIC.txt")
trio_lof = trio_lof[,..to_extract]
trio_lof$source = "trio"
trio_lof = merge(trio_lof, trio_counts, by = "fid", all.x = TRUE)

# if fid in both sib and trio, remove from sib
sib_lof = sib_lof[which(!sib_lof$fid %in% trio_lof$fid),]
all_lof = rbind(unique(trio_lof), sib_lof)
families_missense = merge(families, all_lof, by = "fid", all.x = TRUE)
nrow(families_missense)

In [ ]:
fwrite(families_age_missense, "./gts/aou_gt_missense_ages_added.txt", sep = "\t", 
      row.names = FALSE, quote = FALSE)

In [ ]:
fwrite(families_lof, "./gts/aou_gt_lof_indel_snps_ages_added.txt", sep = "\t", 
      row.names = FALSE, quote = FALSE)